# TRABALHO 8 - COMPUTAÇÃO EVOLUTIVA - PPGGE/UFPA
- Discente: Pedro Henrique do Vale Guimarães
- Docente: Dr. Roberto Célio Limão de Oliveira

Execução de algoritmo genético em representação real.

Dados de execução disponíveis em: 

# 1. Bibliotecas e Parâmetros Aplicados

In [ ]:
# Instalar caso não haja:
# !pip install numpy pandas matplotlib seaborn plotly opencv-python psutil

In [1]:
# Bibliotecas de sistema:
import os
import time
import psutil  
import pickle
import glob
import csv
from concurrent.futures import ProcessPoolExecutor
# bibliotecas de manipulação de dados
import numpy as np
import pandas as pd
# biblioteca do algoritmo genético
import pygad
# bibliotecas de visualização de dados
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  
import seaborn as sns
import plotly.graph_objects as go
import cv2

In [2]:
logs_dir = "logs"
results_dir = "results"
pop_dir = "populations"

os.makedirs(logs_dir, exist_ok=True)
os.makedirs(results_dir, exist_ok=True)
os.makedirs(pop_dir, exist_ok=True)

In [3]:
# Parâmetros Universais:
bound_min, bound_max = -100.0, 100.0  # Domínio da função
POP_SIZE = 500                # tamanho da população inicial
N_GEN = 500                   # número de gerações do algoritmo
N_RUNS = 16                   # número de experimentos 
CROSSOVER_RATE = 0.8          # taxa de cruzamento: 80%
MUTATION_RATE = 0.01          # taxa de mutação: 1%
ELITISM = int(0.01*POP_SIZE)  # indiviuos salvos por elitismo em cada geração (1% da população)
SELECTION = "rws"             # método de seleção de cruzamento: roleta
MUTATION = "random"           # método de mutaçã
CROSSOVER = "uniform"         # método de cruzamento
OBJECTIVE = 'max'             # Objetivo do problema (maximizar ou minimizar)
MULTIPROCESS = None           # Utilização de processamento paralelo

In [4]:
# Função Shaffer Nº 2 (F6):
def f6(x, y):
    num = np.sin(np.sqrt(x**2 + y**2))**2 - 0.5
    denom = (1 + 0.001 * (x**2 + y**2))**2
    return 0.5 - num / denom

In [5]:
# F6 Modficada para 10 variáveis:
def f6_v10(x, y, z, a, b, c, d, e, f, g):
    term = (x**2 + y**2 + z**2 + a**2 + b**2 + c**2 + d**2 + e**2 + f**2 + g**2) 
    num = np.sin(np.sqrt(term))**2 - 0.5
    denom = (1 + 0.001 * (term))**2
    return 0.5 - num / denom

In [8]:
def get_model_config(model_id):
    """
    modelo 1: binário com duas variáveis
    modelo 2: real com duas variáveis
    modelo 3: binário com dez variáveis
    modelo 4: real com dez variáveis
    """
    
    if model_id in [1, 2]:
        NUM_GENES = 2
        
        VAR_BOUNDS = [(bound_min, bound_max)] * NUM_GENES
        fitness_func = f6 
    else:
        NUM_GENES = 10
        
        VAR_BOUNDS = [(bound_min, bound_max)] * NUM_GENES
        fitness_func = f6_v10

    if model_id in [1, 3]:
        REPRESENTATION = 'binary'
        
        GENE_SPACE = [0,1]
        GENE_TYPE = int
        precision = 5   
        N_BITS = int(np.ceil(np.log2((bound_max-bound_min) * np.power(10,precision)))) # número de bits: 25
        GENOME_LENGHT = NUM_GENES * N_BITS    # comprimento do genoma completo 
        
    else: 
        REPRESENTATION = 'real'
        
        GENE_SPACE = {'low': bound_min, 'high': bound_max}
        GENE_TYPE = float
        N_BITS = None
        GENOME_LENGHT = NUM_GENES
    
    return NUM_GENES, REPRESENTATION, VAR_BOUNDS, GENE_SPACE, GENOME_LENGHT, N_BITS, GENE_TYPE, fitness_func

# 2. Funções manuais

In [8]:
def binary_to_float(binary, min_value, max_value):
    """
    Converte uma lista de bits binários para um número float no intervalo [min_value, max_value].

    Parâmetros:
    - binary: lista ou array de 0s e 1s
    - min_value: limite inferior do intervalo
    - max_value: limite superior do intervalo

    Retorna:
    - float no intervalo [min_value, max_value]
    """
    # Converte a lista binária em string e depois em inteiro
    binary_str = ''.join(str(bit) for bit in binary)
    integer_value = int(binary_str, 2)
    
    # Valor máximo possível com o número de bits
    max_int = 2 ** len(binary) - 1

    # Escala linear para [min_value, max_value]
    float_value = min_value + (integer_value / max_int) * (max_value - min_value)
    return float_value


In [9]:
def decode_solution(solution, representation, num_vars, n_bits=None, var_bounds=None):
    """
    Decodifica uma solução com base na representação e número de variáveis.

    Parâmetros:
    - solution: vetor da solução (binária ou real)
    - representation: 'real' ou 'binária'
    - num_vars: número de variáveis (ex: 2 ou 10)
    - n_bits: número de bits por variável (necessário para binária)
    - var_bounds: lista de tuplas (min, max) para cada variável
    """
    if representation == 'real':
        return tuple(solution[:num_vars])

    elif representation == 'binary':
        decoded = []
        for i in range(num_vars):
            start = i * n_bits
            end = (i + 1) * n_bits
            var_bin = solution[start:end]
            var_min, var_max = var_bounds[i]
            var_float = binary_to_float(var_bin, var_min, var_max)
            decoded.append(var_float)
        return tuple(decoded)

    else:
        raise ValueError("Representação inválida. Use 'real' ou 'binary'.")

In [10]:
def dynamic_fitness_function(ga_instance, solution, solution_idx):
    
    decoded = decode_solution(
        solution= solution,
        representation= REPRESENTATION,
        num_vars= NUM_GENES,
        n_bits= N_BITS,
        var_bounds= VAR_BOUNDS
    )
    
    if OBJECTIVE == 'max':
        sign = 1
    elif OBJECTIVE == 'min':
        sign = -1
    else:
        raise ValueError("Representação inválida para objetivo. Use 'max' ou 'min'.")
        
    return sign * fitness_func(*decoded)  # para minimizar

In [11]:
global run_start, last_gen_time

# Função customizada par salvar resultados da geração
def on_generation_custom(ga_instance):
    global last_gen_time

    gen = ga_instance.generations_completed
    fitness_values = ga_instance.last_generation_fitness
    min_fit = np.min(fitness_values)
    mean_fit = np.mean(fitness_values)
    max_fit = np.max(fitness_values)
    
    #process = psutil.Process(os.getpid())
    cpu_percent = process.cpu_percent(interval=None)/psutil.cpu_count()  # Uso de CPU em %
    cpu_time = process.cpu_times()
    total_cpu_seconds = cpu_time.user + cpu_time.system
    mem_info = process.memory_info()
    mem_rss = mem_info.rss / (1024 ** 2)  # Memória em uso (RSS) em MB

    # salvar população completa
    population = ga_instance.population.copy()
    populations[f'gen_{gen}'] = population
    
    now = time.time()

    # tempo para cada geração e total:
    if gen == 1:
        gen_time = now - run_start
        total_time = gen_time
        
    else:
        gen_time = now - last_gen_time
        total_time = now - run_start

    last_gen_time = now
    
    generation_logs.append({
        "run": run,
        "generation": gen,
        "min_fitness": min_fit,
        "mean_fitness": mean_fit,
        "max_fitness": max_fit,
        "time_per_gen": gen_time,
        "total_time": total_time,
        "cpu_percent": cpu_percent,
        "cpu_time": total_cpu_seconds,
        "memory_used_MB": mem_rss,
        })
    
    if gen == 1 or gen % 50 == 0: 
        print(f"Geração {gen:03}: min={min_fit:.5f}, média={mean_fit:.5f}, max={max_fit:.5f}" 
            + f" | CPU: {cpu_percent:.2f}% | RAM: {mem_rss:.2f} MB, Tempo de CPU: {total_cpu_seconds:.3f} s"
        +f" | Tempo/Geração: {gen_time:.3f} s, Tempo Total: {total_time:.3f}")


# 3. Avaliação de utilidade do paralelismo

In [15]:
def dynamic_fitness_function_test(ga_instance, solution, solution_idx):
    # Acessa os parâmetros do modelo diretamente da instância do GA se disponíveis
    # ou precisa passá-los de alguma forma. Usando variáveis globais (não ideal, mas segue o padrão original)
    global CURRENT_REPRESENTATION, CURRENT_NUM_GENES, CURRENT_N_BITS, CURRENT_VAR_BOUNDS, CURRENT_FITNESS_FUNC, CURRENT_OBJECTIVE

    decoded = decode_solution(
        solution=solution,
        representation=CURRENT_REPRESENTATION,
        num_vars=CURRENT_NUM_GENES,
        n_bits=CURRENT_N_BITS,
        var_bounds=CURRENT_VAR_BOUNDS
    )

    if CURRENT_OBJECTIVE == 'max':
        sign = 1
    elif CURRENT_OBJECTIVE == 'min':
        sign = -1
    else:
        raise ValueError("Objetivo inválido. Use 'max' ou 'min'.")

    # Chama a função de fitness correta com os argumentos decodificados
    return sign * CURRENT_FITNESS_FUNC(*decoded)


In [16]:
# Função on_generation simplificada (apenas para permitir a execução)
def on_generation_simple(ga_instance):
    gen = ga_instance.generations_completed
    if gen % 10 == 0 or gen == 1:
      fitness = ga_instance.best_solution()[1]
      #print(f"Gen {gen}: Best Fitness = {fitness:.5f}")
      pass # Apenas para manter a estrutura, não faz log detalhado


In [34]:
# --- Lógica de Teste de Desempenho ---

TEST_N_GEN = 20
TEST_N_RUNS = 1 # Apenas uma execução para medir o tempo
results = []

# Variáveis globais para passar contexto para dynamic_fitness_function
CURRENT_REPRESENTATION = None
CURRENT_NUM_GENES = None
CURRENT_N_BITS = None
CURRENT_VAR_BOUNDS = None
CURRENT_FITNESS_FUNC = None
CURRENT_OBJECTIVE = OBJECTIVE # Usa o objetivo global definido


In [35]:
print(f"Iniciando teste de desempenho com {TEST_N_GEN} gerações e {TEST_N_RUNS} execução(ões).")
print(f"CPUs detectadas: {os.cpu_count()}")

for model_id in range(1, 5):
    NUM_GENES, REPRESENTATION, VAR_BOUNDS, GENE_SPACE, GENOME_LENGTH, N_BITS, GENE_TYPE, fitness_func = get_model_config(model_id)

    # Atualiza variáveis globais de contexto
    CURRENT_REPRESENTATION = REPRESENTATION
    CURRENT_NUM_GENES = NUM_GENES
    CURRENT_N_BITS = N_BITS
    CURRENT_VAR_BOUNDS = VAR_BOUNDS
    CURRENT_FITNESS_FUNC = fitness_func

    print(f"\n--- Modelo {model_id} (Rep: {REPRESENTATION}, Vars: {NUM_GENES}) ---")

    for mode in ['sequential', 'parallel']:
        print(f"  Executando em modo: {mode}")
        parallel_params = None
        if mode == 'parallel':
            # Verifica se há mais de 1 CPU para justificar paralelismo
            cpu_count = os.cpu_count()
            if cpu_count is not None and cpu_count > 1:
                 parallel_params = ["process", cpu_count]
                 print(f"    Usando parallel_processing=['process', {cpu_count}]")
            else:
                 print("Paralelismo não ativado (CPU <= 1 ou contagem indisponível).")
                 # Pula a execução paralela se não for possível/útil
                 continue
        else:
             print("Usando processamento sequencial.")

        start_time = time.time()

        try:
            ga = pygad.GA(
                num_generations=TEST_N_GEN,
                num_parents_mating=int(POP_SIZE * CROSSOVER_RATE),
                fitness_func=dynamic_fitness_function_test,
                sol_per_pop=POP_SIZE,
                num_genes=GENOME_LENGTH,
                gene_space=GENE_SPACE,
                gene_type=GENE_TYPE,
                parent_selection_type=SELECTION,
                crossover_type=CROSSOVER,
                mutation_type=MUTATION,
                keep_elitism=ELITISM,
                mutation_probability=MUTATION_RATE,
                on_generation=on_generation_simple, # Usa a versão simplificada
                parallel_processing=parallel_params,
                # stop_criteria="saturate_10" # Adiciona critério de parada para evitar execuções longas se convergir rápido
            )

            ga.run()

            end_time = time.time()
            duration = end_time - start_time
            results.append([model_id, REPRESENTATION, NUM_GENES, mode, duration])
            print(f"    Tempo de execução ({mode}): {duration:.4f} segundos")

        except Exception as e:
            break

# Salvar resultados em CSV
column_names=['model_id', 'REPRESENTATION', 'NUM_GENES', 'mode', 'duration']
output_file = f"{logs_dir}/sequential-vs-parallel_{TEST_N_GEN}-gens_log.csv"
result_df = pd.DataFrame(results, columns=column_names)
result_df.to_csv(output_file, index=False)
print("Teste de desempenho concluído.")
print(f"\nSalvando resultados em {output_file}")

Iniciando teste de desempenho com 20 gerações e 1 execução(ões).
CPUs detectadas: 4

--- Modelo 1 (Rep: binary, Vars: 2) ---
  Executando em modo: sequential
Usando processamento sequencial.
    Tempo de execução (sequential): 1.8650 segundos
  Executando em modo: parallel
    Usando parallel_processing=['process', 4]
    Tempo de execução (parallel): 13.1622 segundos

--- Modelo 2 (Rep: real, Vars: 2) ---
  Executando em modo: sequential
Usando processamento sequencial.
    Tempo de execução (sequential): 0.6231 segundos
  Executando em modo: parallel
    Usando parallel_processing=['process', 4]
    Tempo de execução (parallel): 2.8764 segundos

--- Modelo 3 (Rep: binary, Vars: 10) ---
  Executando em modo: sequential
Usando processamento sequencial.
    Tempo de execução (sequential): 4.5543 segundos
  Executando em modo: parallel
    Usando parallel_processing=['process', 4]
    Tempo de execução (parallel): 106.2108 segundos

--- Modelo 4 (Rep: real, Vars: 10) ---
  Executando em 

# 4. Execução do algoritmo genético com pygad

In [16]:
# Parâmetros de Reinicialização
model_i = 1         
run_i = 1

for model_id in range(model_i,5): 
    NUM_GENES, REPRESENTATION, VAR_BOUNDS, GENE_SPACE, GENOME_LENGHT, N_BITS, GENE_TYPE, fitness_func = get_model_config(model_id)
    print(f"\nIniciando Modelo {model_id}: Representação: {REPRESENTATION}, Variáveis: {NUM_GENES}")
    best_attempt = {}           # Armazenar melhor resultado por modelo
    global_start = time.time()  # Calcular tempo de execução por modelo
    
    for run in range(run_i, N_RUNS+1):
        print(f"\n Iniciando Execução {run}/{N_RUNS}:")
        run_start = time.time()   # Iniciar tempo do experimento
        last_gen_time = run_start 
        generation_logs = []
        populations = {}
        process = psutil.Process(os.getpid())
        process.cpu_percent(interval=None) # Primeira chamada (descarta)

        ga = pygad.GA(
            num_generations= N_GEN,
            num_parents_mating= int(POP_SIZE * CROSSOVER_RATE), 
            fitness_func = dynamic_fitness_function,
            sol_per_pop = POP_SIZE, 
            num_genes = GENOME_LENGHT,
            gene_space = GENE_SPACE,
            gene_type = GENE_TYPE,
            parent_selection_type = SELECTION,
            crossover_type = CROSSOVER,
            mutation_type = MUTATION,
            keep_elitism = ELITISM,
            mutation_probability = MUTATION_RATE,
            on_generation = on_generation_custom,
            parallel_processing=MULTIPROCESS,
        )

        ga.run() # Execução do AG
        
        # Salva a execução com maior aptidão até o momento:
        fitness_history = ga.best_solutions_fitness

        if not best_attempt:
            try:
                with open(f"{logs_dir}/model_{model_id}_best_run.pkl", "rb") as f:
                    saved_best = pickle.load(f)
                    best_attempt = {
                        "ga": saved_best["ga"],
                        "best_run": saved_best["best_run"],
                        "fitness_history": saved_best["fitness_history"]
                    }
            except:
                best_attempt = {
                    "ga": ga,
                    "best_run": run,
                    "fitness_history": fitness_history
                }
        
        if np.max(fitness_history) > np.max(best_attempt['fitness_history']):
            best_attempt = {
                "ga": ga,
                "best_run": run,
                "fitness_history": fitness_history
            }
            
            with open(f"{logs_dir}/model{model_id}_best_run.pkl", "wb") as f:
                pickle.dump({
                    "ga": best_attempt["ga"],
                    "best_run": best_attempt["best_run"],
                    "fitness_history": best_attempt["fitness_history"]
                }, f)

        np.savez(f"{pop_dir}/model{model_id}_run_{run}_populations.npz", **populations)
        
        # salvar log de cada execução
        df = pd.DataFrame(generation_logs)
        df.to_csv(f"{logs_dir}/model{model_id}_run_{run:02}_logs.csv", index=False)
    
        run_time = time.time() - run_start       # tempo do experimento
        total_time = time.time() - global_start  # tempo total
        
        print(f"Tempo da execução {run}: {run_time:.2f} segundos")
        print(f"Tempo acumulado: {total_time:.2f} segundos")
        print(f"Finalizada Execução {run}/{N_RUNS} — Melhor fitness: {max(fitness_history):.5f}")
        

    
    print(f"\nFinalizado Modelo {model_id}: Representação: {REPRESENTATION}, Variáveis: {NUM_GENES}")


Iniciando Modelo 1: Representação: binary, Variáveis: 2

 Iniciando Execução 1/16:
Geração 001: min=0.07978, média=0.50267, max=0.98508 | CPU: 24.85% | RAM: 325.31 MB, Tempo de CPU: 34.010 s | Tempo/Geração: 0.113 s, Tempo Total: 0.113
Geração 050: min=0.01562, média=0.53591, max=0.99025 | CPU: 24.98% | RAM: 325.31 MB, Tempo de CPU: 37.980 s | Tempo/Geração: 0.090 s, Tempo Total: 4.081
Geração 100: min=0.02468, média=0.85485, max=0.99028 | CPU: 24.48% | RAM: 325.31 MB, Tempo de CPU: 43.840 s | Tempo/Geração: 0.123 s, Tempo Total: 9.953
Geração 150: min=0.00793, média=0.82544, max=0.99028 | CPU: 25.80% | RAM: 328.66 MB, Tempo de CPU: 50.180 s | Tempo/Geração: 0.135 s, Tempo Total: 16.282
Geração 200: min=0.00251, média=0.83880, max=0.99028 | CPU: 26.23% | RAM: 338.20 MB, Tempo de CPU: 56.760 s | Tempo/Geração: 0.133 s, Tempo Total: 23.013
Geração 250: min=0.02157, média=0.84231, max=0.99028 | CPU: 24.40% | RAM: 347.73 MB, Tempo de CPU: 63.340 s | Tempo/Geração: 0.133 s, Tempo Total: 29

# 4. Carregando dados salvos de execução

In [12]:
best_attempts = {}
for model_id in range(1,5):
    with open(f"{logs_dir}/model{model_id}_best_run.pkl", "rb") as f:
        best_attempts[f'model_{model_id}'] = pickle.load(f)

In [13]:
for model_id in range(1,5):
    print(f"Modelo {model_id} - Melhor execução: {best_attempts[f'model_{model_id}']['best_run']}")

Modelo 1 - Melhor execução: 3
Modelo 2 - Melhor execução: 6
Modelo 3 - Melhor execução: 15
Modelo 4 - Melhor execução: 8


In [18]:
log_dfs = {}

for model_id in range(1,5):
    logs_list = sorted(glob.glob(os.path.join(logs_dir, f"model{model_id}_run_*_logs.csv")))
    df_list = [pd.read_csv(f) for f in logs_list]
    df_total = pd.concat(df_list, ignore_index=True)
    df_total.replace([np.inf, -np.inf], np.nan, inplace=True)
    df_total = df_total.interpolate()
    log_dfs[f'model_{model_id}'] = df_total
    

# Visualizações

In [6]:
# Variáveis a serem comparadas
variables = ['min_fitness', 'mean_fitness', 'max_fitness',
             'time_per_gen', 'total_time', 'cpu_percent', 'cpu_time', 'memory_used_MB']

In [15]:
def plot_4model_grid(var):
    combined_df = pd.DataFrame()
    for model in log_dfs:
        df = log_dfs[model].copy()
        df['model'] = model
        combined_df = pd.concat([combined_df, df], ignore_index=True)

    # Plota a variável ao longo das gerações para cada run e modelo
    g = sns.FacetGrid(combined_df, col="model", col_wrap=2, height=4, sharey=True)
    g.map_dataframe(sns.lineplot, x="generation", y=var, hue="run", palette="tab10", linewidth=1)
    g.set_titles(f"{model}")
    g.set_axis_labels("Geração", var)
    g.add_legend(title="Execução")
    plt.subplots_adjust(top=0.9)
    g.fig.suptitle(f"Comparação da variável: {var}")
    plt.savefig(f"{results_dir}/{var}_comparison.png")
    plt.show()

In [25]:
def plot_metrics(dfs, output_dir):
  

    # Concatena dados por modelo e remove infs
    for model in dfs:
        dfs[model].replace([np.inf, -np.inf], np.nan, inplace=True)

    # Para cada variável, plota a comparação entre modelos
    for var in variables:
        fig, axs = plt.subplots(2, 2, figsize=(14, 10), sharex=True)
        axs = axs.flatten()

        for i, model_id in enumerate(sorted(dfs.keys())):
            df = dfs[model_id]
            grouped = df.groupby('generation')[var].agg(['mean', 'min', 'max'])

            ax = axs[i]
            ax.plot(grouped.index, grouped['mean'], label='Média', color='blue')
            ax.fill_between(grouped.index, grouped['min'], grouped['max'], alpha=0.3, label='Min-Max', color='lightblue')
            ax.set_title(f"Modelo {model_id}")
            ax.set_xlabel('Geração')
            ax.set_ylabel(var.replace("_", " ").title())
            ax.legend()
            ax.grid(True)

        # Título geral e layout
        plt.suptitle(f"Comparação entre Modelos para {var.replace('_', ' ').title()}", fontsize=16)
        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        plt.savefig(os.path.join(output_dir, f"{var}_comparison.png"))
        plt.close()

In [26]:
plot_metrics(log_dfs, results_dir)